In [1]:

import os
import numpy as np
import optuna
import pandas as pd
from lightfm import LightFM
from lightfm.cross_validation import random_train_test_split
from lightfm.data import Dataset
from multiprocessing import Pool
import tqdm
from lightfm.evaluation import precision_at_k
import joblib



/home/egor/miniconda3/envs/lightfm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RESULT_PREPROCESSED_PATH = r'/mnt/c/Users/Egor/Desktop/study/diploma/project/dataset_preprocessed'
WIN_RESULT_PREPROCESSED_PATH =r'C:\Users\Egor\Desktop\study\diploma\project\dataset_preprocessed'
SUBMISSION_PATH =r'/mnt/c/Users/Egor/Desktop/study/diploma/project/dataset/sample_submission.csv'

class CFG:
    Submit = True
    Train=True
    OptimizeHyperparams=True

In [3]:
def reduce_mem_usage(df):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024 ** 2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024 ** 2
    print(f'Memory: {start_mem:.2f} Mb -> {end_mem:.2f} Mb ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    return df


def load_data(data_dir):
    print("Загрузка данных...")
    transactions = reduce_mem_usage(pd.read_csv(os.path.join(data_dir, "transactions.csv")))
    articles = reduce_mem_usage(pd.read_csv(os.path.join(data_dir, "articles.csv")))
    customers = reduce_mem_usage(pd.read_csv(os.path.join(data_dir, "customers.csv")))
    rfm_features = reduce_mem_usage(pd.read_csv(os.path.join(data_dir, "rfm_features.csv")))
    return transactions, articles, customers, rfm_features
    

In [4]:
transactions, articles, customers, rfm = load_data(RESULT_PREPROCESSED_PATH)
customers = customers.merge(rfm, on='customer_id', how='left')


Загрузка данных...
Memory: 2197.69 Mb -> 796.66 Mb (63.7% reduction)
Memory: 21.74 Mb -> 13.99 Mb (35.6% reduction)
Memory: 83.74 Mb -> 56.26 Mb (32.8% reduction)
Memory: 83.15 Mb -> 36.38 Mb (56.2% reduction)


In [5]:
user_cat_cols = ['age_group', 'club_member_status', 'fashion_news_frequency']
customers['Active'] = customers['Active'].fillna(0).astype(int).astype(str).apply(lambda x: f"Active_{x}")
customers['FN'] = customers['FN'].fillna(0).astype(int).astype(str).apply(lambda x: f"FN_{x}")
user_cat_cols += ['Active', 'FN']

rfm_cat_cols = []

# recency (дней с последней покупки) – разбиваем на 5 квантилей
customers['recency_cat'] = pd.qcut(customers['recency'].fillna(customers['recency'].median()), 
                                   q=5, labels=['rec_1','rec_2','rec_3','rec_4','rec_5'])
rfm_cat_cols.append('recency_cat')

# frequency (число покупок) – разбиваем на 5 квантилей
customers['frequency_cat'] = pd.qcut(customers['frequency'].fillna(0), 
                                     q=5, labels=['freq_1','freq_2','freq_3','freq_4','freq_5'])
rfm_cat_cols.append('frequency_cat')

# n_unique_articles – квантили
customers['unique_articles_cat'] = pd.qcut(customers['n_unique_articles'].fillna(0), 
                                           q=5, labels=['uniq_1','uniq_2','uniq_3','uniq_4','uniq_5'])
rfm_cat_cols.append('unique_articles_cat')

# fav_product_group – уже категория, заполняем пропуски
customers['fav_product_group'] = customers['fav_product_group'].fillna('Unknown')
rfm_cat_cols.append('fav_product_group')

# Добавляем все RFM-категории в общий список фичей пользователя
user_cat_cols += rfm_cat_cols


In [6]:


item_cat_cols = ['product_group_name', 'colour_group_name']

all_user_ids = customers['customer_id'].unique()
all_article_ids = articles['article_id'].unique()

transactions = transactions[transactions['t_dat'] > '2020-08-21']
transactions = transactions.groupby(['customer_id','article_id']).agg({'price':'sum','t_dat':'count'}).reset_index()
transactions = transactions[['customer_id','article_id','price']]
transactions['price'] = transactions['price'].astype('category')
transactions['price'] = transactions['price'].cat.codes


# Все возможные значения фичей пользователя
user_feature_names = []
for col in user_cat_cols:
    user_feature_names.extend([f"{col}_{v}" for v in customers[col].unique()])

# Все возможные значения фичей товаров
item_feature_names = []
for col in item_cat_cols:
    item_feature_names.extend([f"{col}_{v}" for v in articles[col].unique()])


In [7]:

print(user_feature_names)
print(item_feature_names)


['age_group_40-49', 'age_group_20-29', 'age_group_50-59', 'age_group_30-39', 'age_group_<20', 'age_group_60+', 'club_member_status_ACTIVE', 'club_member_status_UNKNOWN', 'club_member_status_PRE-CREATE', 'club_member_status_LEFT CLUB', 'fashion_news_frequency_NONE', 'fashion_news_frequency_Regularly', 'fashion_news_frequency_Monthly', 'Active_Active_0', 'Active_Active_1', 'FN_FN_0', 'FN_FN_1', 'recency_cat_rec_1', 'recency_cat_rec_2', 'recency_cat_rec_5', 'recency_cat_rec_4', 'recency_cat_rec_3', 'frequency_cat_freq_4', 'frequency_cat_freq_5', 'frequency_cat_freq_1', 'frequency_cat_freq_3', 'frequency_cat_freq_2', 'unique_articles_cat_uniq_4', 'unique_articles_cat_uniq_5', 'unique_articles_cat_uniq_1', 'unique_articles_cat_uniq_3', 'unique_articles_cat_uniq_2', 'fav_product_group_Garment Upper body', 'fav_product_group_Underwear', 'fav_product_group_Garment Lower body', 'fav_product_group_Garment Full body', 'fav_product_group_Swimwear', 'fav_product_group_Accessories', 'fav_product_gro

In [8]:

dataset = Dataset()
dataset.fit(users=all_user_ids,
            items=all_article_ids,
            user_features=user_feature_names,
            item_features=item_feature_names)

def iter_interactions(df_transactions):
    for row in df_transactions.itertuples():
        yield row.customer_id, row.article_id, row.price

interactions_matrix, weights_matrix  = dataset.build_interactions(iter_interactions(transactions))
def build_user_features(dataset, customers_df, cat_cols):
    features = []
    for _, row in customers_df.iterrows():
        uid = row['customer_id']
        feat_vals = [f"{col}_{row[col]}" for col in cat_cols if pd.notna(row[col])]
        features.append((uid, feat_vals))
    return dataset.build_user_features(features, normalize=False)

# Матрица признаков товаров
def build_item_features(dataset, articles_df, cat_cols):
    features = []
    for _, row in articles_df.iterrows():
        aid = row['article_id']
        feat_vals = [f"{col}_{row[col]}" for col in cat_cols if pd.notna(row[col])]
        features.append((aid, feat_vals))
    return dataset.build_item_features(features, normalize=False)

user_feature_matrix = build_user_features(dataset, customers, user_cat_cols)


In [9]:
item_feature_matrix = build_item_features(dataset, articles, item_cat_cols)

print(f"interactions shape: {interactions_matrix.shape}")
print(f"user features shape: {user_feature_matrix.shape}")
print(f"item features shape: {item_feature_matrix.shape}")

interactions shape: (1371980, 105542)
user features shape: (1371980, 1372030)
item features shape: (105542, 105611)


In [10]:
train_interactions, test_interactions  = random_train_test_split(interactions_matrix, test_percentage=0.2, random_state=42)


In [11]:
def objective(trial):
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 0.05, log=True)
    no_components = trial.suggest_int('no_components', 16, 150)
    loss = trial.suggest_categorical('loss', choices=["bpr", "warp", "logistic"])
    item_alpha = trial.suggest_float('item_alpha', 1e-9, 1e-3, log=True)
    user_alpha = trial.suggest_float('user_alpha', 1e-9, 1e-3, log=True)
    max_sampled = trial.suggest_int('max_sampled', 5, 80)
    learning_schedule = trial.suggest_categorical('learning_schedule', choices=["adagrad", "adadelta"])
    
    model = LightFM(no_components=no_components,
                    learning_rate=learning_rate,
                    loss=loss,
                    item_alpha=item_alpha,
                    user_alpha=user_alpha,
                    max_sampled=max_sampled,
                    learning_schedule=learning_schedule,
                    random_state=42)
    
    model.fit(interactions_matrix,
              sample_weight=weights_matrix,
              user_features=user_feature_matrix,
              item_features=item_feature_matrix,
              epochs=20,
              num_threads=20,
              verbose=True)
    prec = precision_at_k(model = model, 
                                    train_interactions = train_interactions, 
                                    test_interactions = test_interactions,
                                    user_features=user_feature_matrix,
                                    item_features=item_feature_matrix,
                                    num_threads=22, 
                                    k=12,
                                    check_intersections = True).mean()
    return prec


if CFG.OptimizeHyperparams:
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=25, show_progress_bar=True)
    joblib.dump(study, "study.pkl")        
    print("Лучшие гиперпараметры:", study.best_params)
    print("Лучшее Precision@12:", study.best_value)

[I 2026-04-27 23:10:42,017] A new study created in memory with name: no-name-c0ab733a-87b3-4a0f-93f2-4732335adbfb
Best trial: 0. Best value: 0.0167526:   4%|██████████████                                                                                                                                                                                                                                                                                                                                                | 1/25 [03:04<1:13:59, 184.97s/it]

[I 2026-04-27 23:13:46,987] Trial 0 finished with value: 0.016752567142248154 and parameters: {'learning_rate': 0.0010253509690168502, 'no_components': 144, 'loss': 'bpr', 'item_alpha': 8.629132190071872e-09, 'user_alpha': 2.2310108018679215e-09, 'max_sampled': 70, 'learning_schedule': 'adadelta'}. Best is trial 0 with value: 0.016752567142248154.



Best trial: 0. Best value: 0.0167526:   8%|████████████████████████████                                                                                                                                                                                                                                                                                                                                  | 2/25 [06:18<1:12:44, 189.77s/it]

[I 2026-04-27 23:17:00,114] Trial 1 finished with value: 1.3074665048407041e-06 and parameters: {'learning_rate': 0.00011364672700011186, 'no_components': 146, 'loss': 'bpr', 'item_alpha': 1.2601639723276828e-08, 'user_alpha': 6.690421166498804e-08, 'max_sampled': 44, 'learning_schedule': 'adagrad'}. Best is trial 0 with value: 0.016752567142248154.



Best trial: 0. Best value: 0.0167526:  12%|██████████████████████████████████████████▏                                                                                                                                                                                                                                                                                                                     | 3/25 [07:42<51:58, 141.75s/it]

...

[I 2026-04-28 00:06:27,151] Trial 22 finished with value: 0.019476022571325302 and parameters: {'learning_rate': 0.04736171481187251, 'no_components': 115, 'loss': 'bpr', 'item_alpha': 9.260178880664067e-07, 'user_alpha': 3.434702137589051e-09, 'max_sampled': 24, 'learning_schedule': 'adadelta'}. Best is trial 11 with value: 0.024630706757307053.



Best trial: 11. Best value: 0.0246307:  96%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████              | 24/25 [58:42<02:35, 155.48s/it]

[I 2026-04-28 00:09:24,810] Trial 23 finished with value: 0.01681401953101158 and parameters: {'learning_rate': 0.0011053515666551404, 'no_components': 136, 'loss': 'bpr', 'item_alpha': 3.904676121710246e-06, 'user_alpha': 2.757217061644604e-08, 'max_sampled': 15, 'learning_schedule': 'adadelta'}. Best is trial 11 with value: 0.024630706757307053.



Best trial: 11. Best value: 0.0246307: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 25/25 [1:01:26<00:00, 147.45s/it]

[I 2026-04-28 00:12:08,284] Trial 24 finished with value: 0.022373365238308907 and parameters: {'learning_rate': 0.0003007817205023727, 'no_components': 121, 'loss': 'bpr', 'item_alpha': 2.532656489084331e-07, 'user_alpha': 1.508009941455626e-09, 'max_sampled': 5, 'learning_schedule': 'adadelta'}. Best is trial 11 with value: 0.024630706757307053.
Лучшие гиперпараметры: {'learning_rate': 0.0011460896324213292, 'no_components': 148, 'loss': 'bpr', 'item_alpha': 5.489516328036859e-09, 'user_alpha': 1.0793576752427113e-09, 'max_sampled': 75, 'learning_schedule': 'adadelta'}
Лучшее Precision@12: 0.024630706757307053


In [12]:
study = joblib.load("study.pkl")
best_hyperparams = study.best_params

final_model = LightFM(**best_hyperparams)
final_model.fit(interactions_matrix,
                sample_weight=weights_matrix,
                user_features=user_feature_matrix,
                item_features=item_feature_matrix,
                epochs=40,
                num_threads=22,
                verbose=True)

prec = precision_at_k(model = final_model, 
                                    train_interactions = train_interactions, 
                                    test_interactions = test_interactions,
                                    user_features=user_feature_matrix,
                                    item_features=item_feature_matrix,
                                    num_threads=22, 
                                    k=12,
                                    check_intersections = True).mean()
print(f"Финальная Precision@12 на подвыборке: {prec:.4f}")


Epoch: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 40/40 [02:25<00:00,  3.65s/it]


Финальная Precision@12 на подвыборке: 0.0311


In [13]:
user_id_map, user_feature_map, item_id_map, item_feature_map = dataset.mapping()
index2item = {v:k for k,v in item_id_map.items()}


In [14]:
def predict_single(user, k=12):
    index = user_id_map[user]
    user_ids = np.repeat(index, len(item_id_map.values()))
    scores = final_model.predict(user_ids, list(item_id_map.values()), item_features=None, user_features=user_feature_matrix, num_threads=1)
    top_items = np.argsort(-scores)[:k]
    article_ids = [index2item[i] for i in top_items]
    return user, article_ids

In [15]:
def create_chunk_indices(meta_df, chunk_idx, chunk_size):
    start_idx = chunk_idx * chunk_size
    end_idx = start_idx + chunk_size
    meta_chunk = meta_df[start_idx:end_idx]
    print("start/end "+str(chunk_idx+1)+":" + str(start_idx) + "," + str(end_idx))
    print(len(meta_chunk))
    return meta_chunk, chunk_idx

def predict_sub_chunks(chunk):
    final_submission=[]
    for row in tqdm.tqdm(chunk[0].values, position=0):
        preds = predict_single(row[0], 12)
        final_submission.append(preds)
      
    return final_submission

In [16]:
import itertools

submission_data = pd.read_csv(SUBMISSION_PATH)
customer_ids = submission_data["customer_id"].unique()



In [17]:
num_cores = 20

def predict_submission(submission_data=None):
    #splitting here by measurement id's to get all signals for a measurement into single chunk
    customer_ids = submission_data["customer_id"].unique()
    df_split = np.array_split(customer_ids, num_cores)
    chunk_size = len(df_split[0])
    all_chunks = []
    for chunk in range(num_cores):
        all_chunks.append(create_chunk_indices(submission_data, chunk, chunk_size))
        
    pool = Pool(num_cores)
    result = pool.map(predict_sub_chunks, all_chunks)
    
    result_combined = list(itertools.chain(result))
    return result_combined

In [18]:
if CFG.Submit:
    final_predictions = predict_submission(submission_data)


start/end 1:0,68599
68599
start/end 2:68599,137198
68599
start/end 3:137198,205797
68599
start/end 4:205797,274396
68599
start/end 5:274396,342995
68599
start/end 6:342995,411594
68599
start/end 7:411594,480193
68599
start/end 8:480193,548792
68599
start/end 9:548792,617391
68599
start/end 10:617391,685990
68599
start/end 11:685990,754589
68599
start/end 12:754589,823188
68599
start/end 13:823188,891787
68599
start/end 14:891787,960386
68599
start/end 15:960386,1028985
68599
start/end 16:1028985,1097584
68599
start/end 17:1097584,1166183
68599
start/end 18:1166183,1234782
68599
start/end 19:1234782,1303381
68599
start/end 20:1303381,1371980
68599


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 68599/68599 [55:48<00:00, 20.49it/s]


In [19]:
flattened = [item for sublist in final_predictions for item in sublist]
print(flattened[0])
flattened = [(i, " ".join(map(str, l))) for i, l in flattened]

('00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657', [828982001, 873679005, 879242006, 664074078, 882760006, 708138037, 816814001, 898860002, 902982001, 780031001, 912100004, 771759011])


In [20]:
df = pd.DataFrame(flattened, columns=['customer_id', 'prediction'])
df.to_csv('result.csv', index=False)